In [1]:
import pandas as pd, numpy as np, pickle, warnings, matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import confusion_matrix, classification_report, roc_curve, auc, accuracy_score, f1_score, roc_auc_score, precision_score, recall_score
warnings.filterwarnings('ignore')
sns.set_theme(style='darkgrid', font_scale=1.1)
print('Imports ✓')

Imports ✓


In [4]:
X_train=pd.read_csv('data/X_train.csv'); X_test=pd.read_csv('data/X_test.csv')
y_train=pd.read_csv('data/y_train.csv').squeeze(); y_test=pd.read_csv('data/y_test.csv').squeeze()
lr=pickle.load(open('models/logistic_model.pkl','rb'))
rf=pickle.load(open('models/rf_model.pkl','rb'))
gbm=pickle.load(open('models/xgb_model.pkl','rb'))
models=[('Logistic Regression',lr),('Random Forest',rf),('XGBoost (GBM)',gbm)]
print('Loaded ✓')

Loaded ✓


In [10]:
for name, m in models:
    print(f"\n{'='*50}\n  {name}\n{'='*50}")
    print(classification_report(y_test, m.predict(X_test), target_names=['No Disease','Disease']))


  Logistic Regression
              precision    recall  f1-score   support

  No Disease       0.90      0.82      0.86        33
     Disease       0.81      0.89      0.85        28

    accuracy                           0.85        61
   macro avg       0.85      0.86      0.85        61
weighted avg       0.86      0.85      0.85        61


  Random Forest
              precision    recall  f1-score   support

  No Disease       0.97      0.88      0.92        33
     Disease       0.87      0.96      0.92        28

    accuracy                           0.92        61
   macro avg       0.92      0.92      0.92        61
weighted avg       0.92      0.92      0.92        61


  XGBoost (GBM)
              precision    recall  f1-score   support

  No Disease       0.93      0.82      0.87        33
     Disease       0.81      0.93      0.87        28

    accuracy                           0.87        61
   macro avg       0.87      0.87      0.87        61
weighted avg     

In [12]:
fig, axes = plt.subplots(1,3,figsize=(15,5))
for ax, (name, m) in zip(axes, models):
    cm = confusion_matrix(y_test, m.predict(X_test))
    pct = cm.astype(float)/cm.sum(axis=1,keepdims=True)*100
    ann = np.array([[f'{v}\n({p:.0f}%)' for v,p in zip(r,rp)] for r,rp in zip(cm,pct)])
    sns.heatmap(cm, annot=ann, fmt='', cmap='Blues', ax=ax, square=True,
                xticklabels=['No Disease','Disease'], yticklabels=['No Disease','Disease'], annot_kws={'size':11,'weight':'bold'})
    ax.set_title(f'{name}\nAcc: {accuracy_score(y_test,m.predict(X_test)):.3f}', fontweight='bold')
    ax.set_xlabel('Predicted'); ax.set_ylabel('Actual')
plt.suptitle('Confusion Matrices — All Models', fontsize=14, fontweight='bold')
plt.tight_layout(); plt.savefig('confusion_matrices.png', bbox_inches='tight'); plt.show()

In [13]:
fig, ax = plt.subplots(figsize=(8,7))
for (name, m), color, ls in zip(models,['#3498db','#2ecc71','#e74c3c'],['-','--','-.']):
    fpr,tpr,_ = roc_curve(y_test, m.predict_proba(X_test)[:,1])
    ax.plot(fpr,tpr,color=color,linestyle=ls,linewidth=2.5,label=f'{name} (AUC={auc(fpr,tpr):.3f})')
ax.plot([0,1],[0,1],'k--',alpha=0.4,label='Random Baseline')
ax.set_xlabel('False Positive Rate'); ax.set_ylabel('True Positive Rate')
ax.set_title('ROC Curves — All Models', fontsize=14, fontweight='bold')
ax.legend(loc='lower right'); plt.tight_layout()
plt.savefig('roc_curves.png', bbox_inches='tight'); plt.show()

In [15]:
feat_cols = pickle.load(open('models/feature_names.pkl','rb'))
fig, axes = plt.subplots(1,2,figsize=(14,5))
for ax, (name, m), color in zip(axes,[('Random Forest',rf),('XGBoost (GBM)',gbm)],['#2ecc71','#e74c3c']):
    imp = pd.Series(m.feature_importances_, index=feat_cols).sort_values(ascending=True)
    ax.barh(imp.index, imp.values, color=color, alpha=0.8)
    ax.set_title(f'{name} — Feature Importance', fontweight='bold')
    ax.set_xlabel('Importance Score')
for i,(v) in enumerate(imp.values):
    axes[1].text(v+.002, i, f'{v:.3f}', va='center', fontsize=8)
plt.tight_layout(); plt.savefig('feature_importance.png', bbox_inches='tight'); plt.show()

In [16]:
rows=[]
for name,m in models:
    p=m.predict(X_test); pr=m.predict_proba(X_test)[:,1]
    rows.append({'Model':name,'Accuracy':f'{accuracy_score(y_test,p):.4f}','Precision':f'{precision_score(y_test,p):.4f}','Recall':f'{recall_score(y_test,p):.4f}','F1':f'{f1_score(y_test,p):.4f}','ROC-AUC':f'{roc_auc_score(y_test,pr):.4f}'})
print(pd.DataFrame(rows).set_index('Model').to_string())
print('\n Phase 3 Complete — Best model: Random Forest (91.8% accuracy, 0.956 AUC)')

                    Accuracy Precision  Recall      F1 ROC-AUC
Model                                                         
Logistic Regression   0.8525    0.8065  0.8929  0.8475  0.9556
Random Forest         0.9180    0.8710  0.9643  0.9153  0.9556
XGBoost (GBM)         0.8689    0.8125  0.9286  0.8667  0.9513

 Phase 3 Complete — Best model: Random Forest (91.8% accuracy, 0.956 AUC)
